In [4]:
from docling.document_converter import DocumentConverter,PdfFormatOption
from docling.chunking import HybridChunker

from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer

import os
import json
import tiktoken

from docling.datamodel.pipeline_options import PdfPipelineOptions
pipeline_options = PdfPipelineOptions()

# VERY IMPORTANT
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = False
# =========================================================
# CONFIG
# =========================================================

PDF_FOLDER = "data/pdf"
OUTPUT_FOLDER = "data/chunks"

MAX_TOKENS = 300

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# =========================================================
# TOKENIZER
# =========================================================

tokenizer = OpenAITokenizer(
    tokenizer=tiktoken.encoding_for_model("gpt-4o"),
    max_tokens=128 * 1024,
)

# =========================================================
# CHUNKER
# =========================================================

chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
    merge_peers=True,
)

# =========================================================
# DOC CONVERTER
# =========================================================

converter = DocumentConverter(
    format_options={
        "pdf": PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# =========================================================
# STORE ALL CHUNKS
# =========================================================

all_chunks = []

# =========================================================
# PROCESS ALL PDFs
# =========================================================

for filename in os.listdir(PDF_FOLDER):

    # Skip non-PDF files
    if not filename.endswith(".pdf"):
        continue

    pdf_path = os.path.join(PDF_FOLDER, filename)

    print(f"\n=================================================")
    print(f"Processing PDF: {filename}")
    print(f"=================================================")

    # =====================================================
    # CONVERT PDF
    # =====================================================

    result = converter.convert(pdf_path)

    # =====================================================
    # EXPORT MARKDOWN
    # =====================================================

    markdown_content = result.document.export_to_markdown()

    markdown_output_path = os.path.join(
        OUTPUT_FOLDER,
        f"{os.path.splitext(filename)[0]}.md"
    )

    with open(markdown_output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    print(f"Markdown exported: {markdown_output_path}")

    # =====================================================
    # EXPORT FULL DOC JSON
    # =====================================================

    doc_dict = result.document.export_to_dict()

    json_output_path = os.path.join(
        OUTPUT_FOLDER,
        f"{os.path.splitext(filename)[0]}.json"
    )

    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(doc_dict, f, indent=2)

    print(f"JSON exported: {json_output_path}")

    # =====================================================
    # CHUNK DOCUMENT
    # =====================================================

    chunks = list(
        chunker.chunk(dl_doc=result.document)
    )

    print(f"Total chunks created: {len(chunks)}")

    # =====================================================
    # PROCESS CHUNKS
    # =====================================================

    processed_chunks = []

    for idx, chunk in enumerate(chunks):

        # -------------------------------------------------
        # SERIALIZED TEXT
        # -------------------------------------------------

        serialized_text = chunker.serialize(chunk=chunk)

        # -------------------------------------------------
        # PAGE NUMBERS
        # -------------------------------------------------

        try:
            page_numbers = sorted(
                list(
                    set(
                        prov.page_no
                        for item in chunk.meta.doc_items
                        for prov in item.prov
                    )
                )
            )
        except Exception:
            page_numbers = []

        # -------------------------------------------------
        # HEADINGS
        # -------------------------------------------------

        headings = (
            chunk.meta.headings
            if chunk.meta.headings
            else []
        )

        # -------------------------------------------------
        # CAPTIONS
        # -------------------------------------------------

        captions = (
            chunk.meta.captions
            if chunk.meta.captions
            else []
        )

        # -------------------------------------------------
        # DOC ITEMS
        # -------------------------------------------------

        try:
            doc_items = [
                str(item.self_ref)
                for item in chunk.meta.doc_items
            ]
        except Exception:
            doc_items = []

        # -------------------------------------------------
        # TOKEN COUNT
        # -------------------------------------------------

        token_count = len(
            tokenizer.tokenizer.encode(serialized_text)
        )

        # -------------------------------------------------
        # ORIGIN METADATA
        # -------------------------------------------------

        try:
            origin_metadata = (
                chunk.meta.origin.model_dump()
                if chunk.meta.origin
                else None
            )
        except Exception:
            origin_metadata = None

        # -------------------------------------------------
        # CHUNK DATA
        # -------------------------------------------------

        chunk_data = {
            "text": serialized_text,

            "metadata": {

                # =========================================
                # BASIC METADATA
                # =========================================

                "filename": filename,
                "chunk_id": idx,

                # =========================================
                # PAGE INFORMATION
                # =========================================

                "page_numbers": (
                    page_numbers
                    if page_numbers
                    else None
                ),

                # =========================================
                # SECTION INFORMATION
                # =========================================

                "title": (
                    headings[0]
                    if headings
                    else None
                ),

                "headings": headings,

                # =========================================
                # TABLE / IMAGE INFO
                # =========================================

                "captions": captions,

                # =========================================
                # DOC REFERENCES
                # =========================================

                "doc_items": doc_items,

                # =========================================
                # TOKEN INFO
                # =========================================

                "token_count": token_count,

                # =========================================
                # ORIGIN INFO
                # =========================================

                "origin": origin_metadata,
            }
        }

        processed_chunks.append(chunk_data)
        all_chunks.append(chunk_data)

    # =====================================================
    # SAVE PDF-SPECIFIC CHUNKS
    # =====================================================

    chunk_output_path = os.path.join(
        OUTPUT_FOLDER,
        f"{os.path.splitext(filename)[0]}_chunks.json"
    )

    with open(chunk_output_path, "w", encoding="utf-8") as f:
        json.dump(processed_chunks, f, indent=2)

    print(f"Chunk JSON exported: {chunk_output_path}")

# =========================================================
# SAVE ALL CHUNKS
# =========================================================

all_chunks_output_path = os.path.join(
    OUTPUT_FOLDER,
    "all_chunks.json"
)

with open(all_chunks_output_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2)

print("\n=================================================")
print("ALL PDFs PROCESSED SUCCESSFULLY")
print("=================================================")
print(f"Total chunks across all PDFs: {len(all_chunks)}")
print(f"Combined chunks saved at: {all_chunks_output_path}")


Processing PDF: NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 3655.46it/s]


Markdown exported: data/chunks\NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.md
JSON exported: data/chunks\NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.json
Total chunks created: 18
Chunk JSON exported: data/chunks\NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper_chunks.json

Processing PDF: paper1.pdf


C:\Users\Nitin.Vedwal\AppData\Local\Temp\ipykernel_3864\3623276451.py:140: DeprecationWarning: Use contextualize() instead.
  serialized_text = chunker.serialize(chunk=chunk)
C:\Users\Nitin.Vedwal\AppData\Local\Temp\ipykernel_3864\3623276451.py:175: DeprecationWarning: deprecated
  if chunk.meta.captions


Markdown exported: data/chunks\paper1.md
JSON exported: data/chunks\paper1.json
Total chunks created: 27
Chunk JSON exported: data/chunks\paper1_chunks.json

Processing PDF: Understanding_Climate_Change.pdf


C:\Users\Nitin.Vedwal\AppData\Local\Temp\ipykernel_3864\3623276451.py:140: DeprecationWarning: Use contextualize() instead.
  serialized_text = chunker.serialize(chunk=chunk)
C:\Users\Nitin.Vedwal\AppData\Local\Temp\ipykernel_3864\3623276451.py:175: DeprecationWarning: deprecated
  if chunk.meta.captions
Stage preprocess failed for run 3, pages [22]: std::bad_alloc
Stage preprocess failed for run 3, pages [23]: std::bad_alloc
Stage preprocess failed for run 3, pages [24]: std::bad_alloc
Stage preprocess failed for run 3, pages [25]: std::bad_alloc
Stage preprocess failed for run 3, pages [26]: std::bad_alloc
Stage preprocess failed for run 3, pages [27]: std::bad_alloc
Stage preprocess failed for run 3, pages [28]: std::bad_alloc
Stage preprocess failed for run 3, pages [29]: std::bad_alloc
Stage preprocess failed for run 3, pages [30]: std::bad_alloc
Stage preprocess failed for run 3, pages [31]: std::bad_alloc
Stage preprocess failed for run 3, pages [32]: std::bad_alloc
Stage prepro

Markdown exported: data/chunks\Understanding_Climate_Change.md
JSON exported: data/chunks\Understanding_Climate_Change.json
Total chunks created: 138
Chunk JSON exported: data/chunks\Understanding_Climate_Change_chunks.json

ALL PDFs PROCESSED SUCCESSFULLY
Total chunks across all PDFs: 183
Combined chunks saved at: data/chunks\all_chunks.json


C:\Users\Nitin.Vedwal\AppData\Local\Temp\ipykernel_3864\3623276451.py:140: DeprecationWarning: Use contextualize() instead.
  serialized_text = chunker.serialize(chunk=chunk)
C:\Users\Nitin.Vedwal\AppData\Local\Temp\ipykernel_3864\3623276451.py:175: DeprecationWarning: deprecated
  if chunk.meta.captions
